In [10]:
pip install scikit-rf

In [11]:
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- S2P loader (robust option) ----
# Uses scikit-rf to parse S2P properly.
import skrf as rf

import torch
import torch.nn as nn
import torch.optim as optim

import os, glob
from torch.utils.data import Dataset, DataLoader

In [12]:
def load_s21_mag_db_from_s2p(path: str):
    ntwk = rf.Network(path)

    f_hz = ntwk.f  # shape (M,)
    s = ntwk.s     # shape (M, 2, 2), complex

    # S21 is row=1, col=0 in 0-based indexing (port2 <- port1)
    s21 = s[:, 1, 0]
    mag = np.abs(s21)
    mag_db = 20.0 * np.log10(np.maximum(mag, 1e-12))

    return f_hz.astype(np.float64), mag_db.astype(np.float64)

In [13]:
def resample_to_fixed_grid(f_hz: np.ndarray, y: np.ndarray, n_points: int):
    # Create uniform grid over existing range and interpolate
    f_min, f_max = float(f_hz.min()), float(f_hz.max())
    f_new = np.linspace(f_min, f_max, n_points)
    y_new = np.interp(f_new, f_hz, y)
    return f_new, y_new

In [14]:
def standardize_1d(y: np.ndarray):
    mu = float(y.mean())
    sigma = float(y.std() + 1e-12)
    y_std = (y - mu) / sigma
    return y_std, mu, sigma

In [15]:
class Conv1DAutoencoder(nn.Module):
    def __init__(self, latent_channels=16):
        super().__init__()
        # Input: (B, C=1, N)

        # Encoder
        self.enc = nn.Sequential(
            nn.Conv1d(1, 8, kernel_size=9, padding=4),
            nn.ReLU(),
            nn.MaxPool1d(2),  # N -> N/2

            nn.Conv1d(8, 16, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),  # N -> N/4

            nn.Conv1d(16, latent_channels, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),  # N -> N/8
        )

        # Pooling head for fixed-length features: (B, C, L) -> (B, C)
        self.pool = nn.AdaptiveAvgPool1d(1)

        # Decoder (mirror)
        self.dec = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="nearest"),  # N/8 -> N/4
            nn.Conv1d(latent_channels, 16, kernel_size=5, padding=2),
            nn.ReLU(),

            nn.Upsample(scale_factor=2, mode="nearest"),  # N/4 -> N/2
            nn.Conv1d(16, 8, kernel_size=7, padding=3),
            nn.ReLU(),

            nn.Upsample(scale_factor=2, mode="nearest"),  # N/2 -> N
            nn.Conv1d(8, 1, kernel_size=9, padding=4),
            # No activation: reconstruction in standardized space can be any real number
        )

    def encode(self, x, return_map: bool = False):
        """
        Returns:
          - latent map z with shape (B, latent_channels, L) if return_map=True
          - pooled feature vector feat with shape (B, latent_channels) otherwise
        """
        z = self.enc(x)  # (B, C, L)
        if return_map:
            return z
        feat = self.pool(z).squeeze(-1)  # (B, C, 1) -> (B, C)
        return feat

    def forward(self, x):
        z = self.enc(x)
        y = self.dec(z)
        return y

In [16]:
def prepare_dataset(s2p_files, n_points=1024):
    signals = []

    for file in s2p_files:
        try:
            freq, mag = load_s21_mag_db_from_s2p(file)
            freq, mag = resample_to_fixed_grid(freq, mag, n_points)
            mag, _, _ = standardize_1d(mag)
            signals.append(mag)
        except:
            continue

    signals = np.array(signals)  # (N_samples, 1024)
    return torch.tensor(signals).float()

In [17]:
def extract_features_from_files(
    s2p_files,
    checkpoint_path,
    n_points=1024,
    latent_channels=16,
    batch_size=64,
    device=None,
    return_metadata=False
):
    """
    Extracts feature vectors using the trained Conv1DAutoencoder encoder.

    Returns:
      - feats: np.ndarray of shape (N, latent_channels)
      - paths: list[str] of length N (only if return_metadata=True)
      - norm_stats: list[tuple(mu, sigma)] of length N (only if return_metadata=True)
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1) Build model & load weights
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = Conv1DAutoencoder(latent_channels=latent_channels).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # 2) Preprocess each file -> tensor (N, 1, n_points)
    sensor = []
    volunteer = []
    xs = []
    kept_paths = []
    norm_stats = []

    for p in s2p_files:
        try:
            f_hz, mag_db = load_s21_mag_db_from_s2p(p)
            _, mag_db = resample_to_fixed_grid(f_hz, mag_db, n_points)
            mag_std, mu, sigma = standardize_1d(mag_db)

            sensor.append(p.split('/')[-3])
            volunteer.append(p.split('/')[-4].replace(' ','_'))
            xs.append(mag_std.astype(np.float32))
            kept_paths.append(p)
            norm_stats.append((mu, sigma))
        except Exception:
            # Skip unreadable/bad files (you can log them if you want)
            continue

    if len(xs) == 0:
        raise ValueError("No valid S2P files could be processed.")

    X = torch.tensor(np.stack(xs, axis=0)).unsqueeze(1).to(device)  # (N,1,Npoints)

    # 3) Encode in batches
    feats_list = []
    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = X[i:i+batch_size]
            fb = model.encode(xb, return_map=False)  # (B, latent_channels)
            feats_list.append(fb.cpu())

    feats = torch.cat(feats_list, dim=0).numpy()

    # (N, latent_channels)# feats is (N, latent_channels)
    feat_cols = [f"feat_{i}" for i in range(feats.shape[1])]
    feat_df = pd.DataFrame(feats, columns=feat_cols)

    # split norm_stats into mu, sigma columns
    mu_sigma = np.array(norm_stats, dtype=np.float64)  # shape (N, 2)
    meta_df = pd.DataFrame({
        "volunteer": volunteer,
        "device": sensor,
        "kept_paths": kept_paths,
        "mu": mu_sigma[:, 0],
        "sigma": mu_sigma[:, 1],
        })

    # final feature table
    features = pd.concat([meta_df.reset_index(drop=True), feat_df.reset_index(drop=True)], axis=1)
    return features

In [18]:
root_dir = '/content/drive/MyDrive/Data/MAS Volunteer Study March 2022'

print(f"Traversing directory: {root_dir}")

s2p_files = []
for dirpath, dirnames, filenames in os.walk(root_dir):
    # if 'SRR' in dirpath:
    #     continue
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)
        if filename == "Air.s2p":
          continue
        else:
          s2p_files.append(file_path)

Traversing directory: /content/drive/MyDrive/Data/MAS Volunteer Study March 2022


In [20]:
chk_point_a = "/content/drive/MyDrive/Data/checkpoints/autoencoder/checkpoint_epoch_191.pt"
chk_point_de_gauss = "/content/drive/MyDrive/Data/checkpoints/de_autoencoder/gaussian/checkpoint_epoch_191.pt"
chk_point_de_mask = "/content/drive/MyDrive/Data/checkpoints/de_autoencoder/mask/checkpoint_epoch_191.pt"
chk_point_de_comb = "/content/drive/MyDrive/Data/checkpoints/de_autoencoder/combined/checkpoint_epoch_191.pt"
features_autoencoder = extract_features_from_files(s2p_files, chk_point_a)
features_denoising_gaussian = extract_features_from_files(s2p_files, chk_point_de_gauss)
features_denoising_mask = extract_features_from_files(s2p_files, chk_point_de_mask)
features_denoising_comb = extract_features_from_files(s2p_files, chk_point_de_comb)


In [21]:
output_dir = '/content/drive/MyDrive/Data/prepared data'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'features_autoencoder.csv')
features_autoencoder.to_csv(output_path, index=False)
print(f"Features DataFrame saved to {output_path}")

Features DataFrame saved to /content/drive/MyDrive/Data/prepared data/features_autoencoder.csv


In [22]:
output_dir = '/content/drive/MyDrive/Data/prepared data'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'features_denoising_gaussian.csv')
features_denoising_gaussian.to_csv(output_path, index=False)
print(f"Features DataFrame saved to {output_path}")

Features DataFrame saved to /content/drive/MyDrive/Data/prepared data/features_denoising_gaussian.csv


In [23]:
output_dir = '/content/drive/MyDrive/Data/prepared data'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'features_denoising_mask.csv')
features_denoising_mask.to_csv(output_path, index=False)
print(f"Features DataFrame saved to {output_path}")

Features DataFrame saved to /content/drive/MyDrive/Data/prepared data/features_denoising_mask.csv


In [24]:
output_dir = '/content/drive/MyDrive/Data/prepared data'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'features_denoising_comb.csv')
features_denoising_comb.to_csv(output_path, index=False)
print(f"Features DataFrame saved to {output_path}")

Features DataFrame saved to /content/drive/MyDrive/Data/prepared data/features_denoising_comb.csv
